# 03 — Prepare (feature engineering) & Export

The analysis-ready stage: take the cleaned `speeches_clean` table and **derive the
features** the analysis + charts need, persist them to `data/processed/`, and package
the sellable export + codebook. (`02-clean` only tidied/typed the raw data; this is
where metrics and derived tables are built.)

Feature logic lives in `src/prepare.py` (transparent tokenizer, pronoun counts,
speech-type classifier, pure-Python TF-IDF, tenure builder). Tables built here:
- **`speeches_features`** — one row per speech + `word_count`, self/collective counts
  and per-1k rates, `self_share`, `speech_type`, `is_sotu_series`, `delivery_mode`.
- **`president_terms`** — days in office → tenure-weighted `speeches_per_year`
  (`reliable_rate` flags <1yr tenures).
- **`word_freq_by_president`** — top common words per president (stopwords removed).
- **`distinctive_words_by_president`** — TF-IDF defining vocabulary per president.

**Apples-to-oranges guard:** `speech_type` + `is_sotu_series` are what let `04-viz`
compare like-with-like (SOTU vs SOTU); `delivery_mode` marks the written(≤1912)/spoken break.

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd
from datetime import date

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.ingest import load_config
from src.clean_quality import (
    get_connection, run_sql, quality_report, save_processed, load_to_duckdb,
    register_source, get_sources,
)
from src.prepare import (
    add_speech_metrics, build_president_terms, stopwords,
    top_words_by_group, distinctive_words_by_group, package_dataset,
)

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')

## Load cleaned speeches

In [ ]:
clean = run_sql('SELECT * FROM speeches_clean', con)
print('speeches_clean:', clean.shape)
clean.head(2)

## Feature table 1 — `speeches_features` (per-speech metrics + speech type)

`add_speech_metrics()` derives word count, self (I/me/my/mine) vs collective
(we/us/our/ours) counts + per-1k rates + `self_share`, and classifies each speech's
institutional type. `delivery_mode` marks the written(clerk-read, 1801–1912) vs
spoken era — load-bearing for the SOTU trend.

In [ ]:
feat = add_speech_metrics(clean, text_col='transcript', title_col='title')
feat['delivery_mode'] = feat['year'].map(lambda y: 'written_era' if 1801 <= y <= 1912 else 'spoken_era')
print(feat.shape)
print('speech types:', feat['speech_type'].value_counts().to_dict())
feat[['president','year','speech_type','is_sotu_series','word_count',
      'self_per_1k','collective_per_1k','self_share']].head(6)

In [ ]:
load_to_duckdb(feat, 'speeches_features', con)
save_processed(feat, cfg, 'speeches_features.parquet')
register_source(
    con, table='speeches_features',
    name='Miller Center speeches + derived language metrics',
    url='https://data.millercenter.org/miller_center_speeches.tgz',
    license='Public domain (U.S. government works)',
    notes='Per-speech word_count, self/collective pronoun counts+rates, self_share, speech_type, delivery_mode. Derived in 03-prepare from speeches_clean.',
    methodology='Transparent lowercase word-regex tokenizer (HTML entities decoded; contractions I\'m/we\'ll/let\'s attributed to their pronoun). speech_type from Miller Center title.',
    series_breaks='delivery_mode: written era 1801-1912 (clerk-read) vs spoken; not directly comparable on raw pronoun density.',
)

## Feature table 2 — `president_terms` (tenure weighting)

From the curated `data/raw/president_terms.csv` (inauguration/exit dates — a
non-regenerable reference input, committed). `build_president_terms()` sums days in
office (non-consecutive Cleveland/Trump handled), joins corpus speech counts, and
derives `speeches_per_year`. `reliable_rate` = tenure ≥ 1yr (excludes W. Harrison
31d / Garfield 199d tiny-denominator artifacts). NOTE: speeches_per_year is a
CORPUS-COVERAGE rate (curated inclusion ÷ tenure), not true speaking output.

In [ ]:
terms = build_president_terms('data/raw/president_terms.csv', clean)
missing = set(clean.president) - set(terms.president)
assert not missing, f'presidents missing from term table: {missing}'
load_to_duckdb(terms, 'president_terms', con)
save_processed(terms, cfg, 'president_terms.parquet')
register_source(
    con, table='president_terms',
    name='U.S. presidential term dates (curated public record)',
    url='public record (inauguration/exit dates)', license='Public domain (historical fact)',
    notes='days_in_office + tenure-weighted speeches_per_year. Non-consecutive terms summed; Trump 2nd term capped at retrieval date.',
    methodology='Hand-curated from uncontested public record.',
    series_breaks='reliable_rate=False for <1yr tenures (tiny-denominator rate artifact).',
)
print(terms.sort_values('years_in_office').head(3).to_string(index=False))

## Feature tables 3 & 4 — word frequencies + TF-IDF distinctiveness

`word_freq_by_president` = common words (raw frequency, stopwords removed);
`distinctive_words_by_president` = TF-IDF (words a president uses far more than
other presidents = their defining vocabulary). Both feed the word clouds / picker.

In [ ]:
print('stopwords:', len(stopwords()))
word_freq = top_words_by_group(clean, 'president', 'transcript', top_n=60)
distinctive = distinctive_words_by_group(clean, 'president', 'transcript', top_n=60)
load_to_duckdb(word_freq, 'word_freq_by_president', con)
load_to_duckdb(distinctive, 'distinctive_words_by_president', con)
save_processed(word_freq, cfg, 'word_freq_by_president.parquet')
save_processed(distinctive, cfg, 'distinctive_words_by_president.parquet')
for p in ['Abraham Lincoln','Franklin D. Roosevelt','Ronald Reagan']:
    print(f'{p:24s}', distinctive[distinctive.president==p].head(6).word.tolist())

## Quality report + package the export

Package the per-speech feature table as the sellable dataset (CSV + Excel + Parquet
+ codebook). Drop the heavy `transcript` text from the export — the value is the
derived metrics; the full text stays reproducible from the pipeline.

In [ ]:
quality_report(feat, table_name='speeches_features', con=con,
               required_columns=['president','year','speech_type','word_count',
                                 'self_count','collective_count'], max_null_pct=0.05)
con.execute('DROP TABLE IF EXISTS _qc_speeches_features')

export_df = feat.drop(columns=['transcript'])
codebook = {
    'president': 'U.S. president who delivered the speech.',
    'speech_date': 'Date of delivery (YYYY-MM-DD).',
    'year': 'Year of delivery.',
    'title': 'Miller Center speech title.',
    'url': 'Source URL at millercenter.org.',
    'source_file': 'Original JSON filename in the Miller Center archive.',
    'word_count': 'Total word tokens (transparent lowercase tokenizer, HTML entities decoded).',
    'self_count': 'Count of self pronouns: I, me, my, mine, myself (+ contractions I\'m/I\'ve/I\'ll/I\'d).',
    'collective_count': 'Count of collective pronouns: we, us, our, ours, ourselves (+ we\'re/we\'ve/we\'ll/we\'d/let\'s).',
    'self_per_1k': 'self_count per 1,000 words.',
    'collective_per_1k': 'collective_count per 1,000 words.',
    'self_share': 'self_count / (self_count + collective_count). 0.5 = balanced; null if neither present.',
    'speech_type': 'Institutional type from the title (Inaugural Address, State of the Union, Annual Message, etc.).',
    'is_sotu_series': 'True if State of the Union OR (pre-1929) Annual Message — the unified SOTU series.',
    'delivery_mode': 'written_era (1801-1912 clerk-read messages) vs spoken_era — a comparability break.',
}
notes = ('Source: Miller Center (UVA) presidential speech archive, public domain. '
         'CURATED corpus of major speeches (not exhaustive); coverage denser for modern '
         'presidents. Measures the speech AS DELIVERED (many were ghostwritten). '
         'Pronoun/word metrics from a transparent tokenizer (see src/prepare.py).')
package_dataset(export_df, cfg, name='presidential_speeches_v1', codebook=codebook, notes=notes)

In [ ]:
print('tables now:', [r[0] for r in con.execute('SHOW TABLES').fetchall()])
get_sources(con)[['duckdb_table','source_name']]

---
**Next:** `04-viz.ipynb` — explore self/collective, word counts, tenure, and the
per-president word clouds, reading from these processed feature tables.

---
## Cleanup
Close the DuckDB connection so the lock is released for other tools.

In [ ]:
con.close()
print('connection closed')